# PBIP Data Generation

In [1]:
# Change source dir to project root
%cd ..

/data/pere.carrillo/pbi


/home/pere.carrillo/micromamba/envs/pbi-finetune/lib/python3.10/site-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [2]:
import os
import pandas as pd
from pathlib import Path
from Bio import SeqIO

In [ ]:
BASE_DIR = Path("data/PBIP")

# DATASET = "pbip_test"
# DATASET = "pbip_train"
DATASET = "predphi_train"
# DATASET = "predphi_test"

BAC_FASTA_DIR = BASE_DIR / "bac_faa_2021"
PHAGE_FASTA_DIR = BASE_DIR / "phage_faa_2021"
INTERACTIONS_CSV = BASE_DIR / f"{DATASET}_dataset.csv"

In [4]:
OUT_DIR = BASE_DIR / f"perphect_format/{DATASET}"
os.makedirs(OUT_DIR, exist_ok=True)

In [5]:
interactions = pd.read_csv(INTERACTIONS_CSV)

interactions = interactions.rename(columns={
    "phage": "phage_id",
    "host": "bacterium_id",
    "label": "interaction_type"
})

interactions.head()

,phage_id,bacterium_id,interaction_type
0,KM236502,CP027541,1
1,NC_031124,CP027541,1
2,NC_005357,MTKC01000003,0
3,MH651183,CP027541,1
4,MH051257,CP020567,0


In [6]:
bacteria_ids = set(interactions["bacterium_id"])
print("Unique bacteria in interactions:", len(bacteria_ids))

bact_records = []

for bac_id in bacteria_ids:
    fasta_path = BAC_FASTA_DIR / f"{bac_id}.fasta"

    if not fasta_path.exists():
        print(f"[WARNING] Missing bacterial FASTA: {bac_id}")
        continue

    # Read ALL sequences in the file (all contigs)
    seqs = [str(rec.seq) for rec in SeqIO.parse(str(fasta_path), "fasta")]

    if len(seqs) == 0:
        print(f"[WARNING] Empty FASTA: {bac_id}")
        continue

    # Concatenate all contigs into one sequence
    merged_seq = "".join(seqs)
    length = len(merged_seq)

    bact_records.append((bac_id, merged_seq, length))

bacteria_df = pd.DataFrame(
    bact_records,
    columns=["bacterium_id", "bacterium_sequence", "sequence_length"]
)

print(f"Total bacteria loaded: {len(bacteria_df)}")
bacteria_df.head()

Unique bacteria in interactions: 194
[WARNING] Missing bacterial FASTA: CP020358
[WARNING] Missing bacterial FASTA: LR134359
[WARNING] Missing bacterial FASTA: CP027541
[WARNING] Missing bacterial FASTA: NZ_CDQJ01000001
[WARNING] Missing bacterial FASTA: CP035765
[WARNING] Missing bacterial FASTA: CP017065
[WARNING] Missing bacterial FASTA: QOUU01000001
[WARNING] Missing bacterial FASTA: NZ_OCQN01000001
[WARNING] Missing bacterial FASTA: RKJP01000005
[WARNING] Missing bacterial FASTA: NZ_KV917376
[WARNING] Missing bacterial FASTA: NZ_JNIU01000001
[WARNING] Missing bacterial FASTA: CP020369
[WARNING] Missing bacterial FASTA: CP029760
[WARNING] Missing bacterial FASTA: CP024679
[WARNING] Missing bacterial FASTA: CP025095
[WARNING] Missing bacterial FASTA: CP023483
[WARNING] Missing bacterial FASTA: LR134303
[WARNING] Missing bacterial FASTA: MH400164
[WARNING] Missing bacterial FASTA: NZ_AEUT02000001
[WARNING] Missing bacterial FASTA: NZ_LT629770
[WARNING] Missing bacterial FASTA: NZ_FNO

,bacterium_id,bacterium_sequence,sequence_length


In [44]:
bacteria_df.to_csv(OUT_DIR/"bacteria_df.csv", index=False)

In [45]:
phage_records = []

phage_ids = set(interactions["phage_id"])
print(f"Unique phages in interactions: {len(phage_ids)}")

for ph_id in phage_ids:
    fasta_path = PHAGE_FASTA_DIR / f"{ph_id}.fasta"
    if not fasta_path.exists():
        print(f"[WARNING] Missing phage FASTA: {ph_id}")
        continue
    
    for rec in SeqIO.parse(str(fasta_path), "fasta"):
        seq = str(rec.seq)
        length = len(rec.seq)
        phage_records.append((ph_id, seq, length))

phages_df = pd.DataFrame(
    phage_records,
    columns=["phage_id", "phage_sequence", "sequence_length"]
)

print(f"Total phages loaded: {len(phages_df)}")
phages_df.head()

Unique phages in interactions: 83
Total phages loaded: 83


,phage_id,phage_sequence,sequence_length
0,PH-KP9074-1,ATTTTCCGAGGCTTACACGCAGGCGCGCCAGCTTTACGAGCAAATG...,48792
1,PH-KP8632,GGTGCGCAGAGCCGCGGTGGAAGAAAATGCGGCGAAATCCCCATAG...,58924
2,PHKP9471-A1,TAGCAGTGCCCCCCCCCCCCCTAAATTGACGCTAGGCACCCCCTAT...,41141
3,PH-KP9332_A1,CGCGAATGACGGTGAAGCGCCACTTGCCTAACCAGTGGAAGTAAAC...,49506
4,PH-KP9628-B1,TTCTCTTTCGCCGCCTGCTCTTTCGCGGCTTGTTCTTCCGCATCGA...,58597


In [46]:
phages_df.to_csv(OUT_DIR/"phages_df.csv", index=False)

In [47]:
interactions.to_csv(OUT_DIR/"couples_df.csv", index=False)